# 🚀 EL ÁLBUM B — V9.8 MASTER
### Flujo completo: Instalación → Motor → Procesado → Real-ESRGAN
---
⚠️ **IMPORTANTE: Activa la GPU antes de empezar**
Entorno de ejecución → Cambiar tipo de entorno de ejecución → **T4 GPU** → Guardar

Luego ejecuta las celdas **en orden, de arriba a abajo**.

## CELDA 1 — Instalación de dependencias
Ejecuta **una vez por sesión**. Reinicia la sesión cuando termine.

In [ ]:
import sys

# Verificar GPU activa
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    print('❌ GPU no detectada. Ve a: Entorno de ejecución → Cambiar tipo → T4 GPU')
else:
    print('✅ GPU detectada:', result.stdout.split('\n')[8].strip())

# Instalar dependencias con versiones compatibles
!{sys.executable} -m pip install "numpy>=2.3.0" --upgrade -q
!{sys.executable} -m pip uninstall onnxruntime onnxruntime-gpu -y -q
!{sys.executable} -m pip install "onnxruntime==1.19.2" -q
!{sys.executable} -m pip install "rembg==2.0.76" huggingface_hub Pillow -q

print('\n✅ Dependencias instaladas.')
print('👉 Ahora: Entorno de ejecución → Reiniciar sesión → luego ejecuta Celda 2')

## CELDA 2 — Definición del motor V9.8
Carga la clase en memoria. **No procesa nada todavía.**

Correcciones incluidas:
- `[FIX-6]` Alpha suavizado con multiply para eliminar escaleras en chaquetas/trajes oscuros
- `[FIX-7]` Sharpening + boost de color aplicado sobre sujetos antes de componer

In [ ]:
import os
import glob
import numpy as np
from PIL import Image, ImageEnhance, ImageFilter, ImageChops

try:
    import rembg
    _new_session = getattr(rembg, 'new_session', None)
    if _new_session is None:
        from rembg.session_factory import new_session as _new_session
    from rembg import remove
    from google.colab import drive, userdata
    from huggingface_hub import InferenceClient
    print('✅ Librerías cargadas correctamente.')
except ModuleNotFoundError as e:
    print(f'❌ Falta instalar: {e}. Vuelve a ejecutar la Celda 1.')
    raise SystemExit


class ElAlbumB_V9_8_Master:
    def __init__(self):
        print('\n🔗 Conectando con Google Drive...')
        drive.mount('/content/drive', force_remount=True)
        self.ruta_base = '/content/drive/MyDrive/ElAlbumB_Clientes/'
        self.session = _new_session('u2net_human_seg')
        try:
            hf_token = userdata.get('HF_TOKEN')
            self.client = InferenceClient(
                model='black-forest-labs/FLUX.1-schnell',
                token=hf_token,
                timeout=120
            )
            print('✅ Conexión con FLUX.1 establecida.')
        except Exception as e:
            print('❌ Error de autenticación con HF_TOKEN.')
            raise e

        self.banco_prompts = {
            'finca': 'Wide-angle professional studio photography of beautiful rolling green hills, clean vibrant sunset sky, sharp focus, rich natural colors, cinematic lighting, 8k, no people',
            'vogue': 'Wide-angle luxury modern villa open terrace overlooking a breathtaking coastal ocean sunset, professional editorial photography, sharp focus, vibrant golden hour, no people',
            'nostalgia': 'Wide-angle professional photography of an empty charming private colonial courtyard at sunset, warm terracotta walls, romantic golden ambient light, sharp focus, 8k, no people',
            'magic': 'Wide-angle editorial landscape of a mystical beautiful green valley during twilight, soft glowing lighting, dramatic sky with intense warm clouds, sharp focus, no people',
            'cinematic': 'Wide-angle dramatic cinematic landscape framing, rolling hills under an epic saturated twilight sunset sky, deep rich contrast, professional photography, 8k, no people',
        }

    def _calcular_dimensiones_flux(self, ancho_orig, alto_orig):
        proporcion = ancho_orig / alto_orig
        if proporcion >= 1.0:
            ancho_flux, alto_flux = 1024, int(1024 / proporcion)
        else:
            alto_flux, ancho_flux = 1024, int(1024 * proporcion)
        return (ancho_flux // 16) * 16, (alto_flux // 16) * 16

    def _inyectar_micro_grano_fondo(self, img_pil, intensidad=3.0):
        arr = np.array(img_pil).astype(np.float32)
        luma = 0.299 * arr[..., 0] + 0.587 * arr[..., 1] + 0.114 * arr[..., 2]
        modulador = np.clip(1.0 - (np.abs(luma - 120.0) / 150.0), 0.2, 1.0)
        ruido = np.random.normal(0, intensidad, luma.shape) * modulador
        for i in range(3):
            arr[..., i] += ruido
        return Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))

    def _aplicar_high_pass_sharpen(self, img_pil, radio=2, fuerza=1.1):
        arr_orig = np.array(img_pil).astype(np.float32)
        arr_blur = np.array(img_pil.filter(ImageFilter.GaussianBlur(radius=radio))).astype(np.float32)
        return Image.fromarray(np.clip(arr_orig + fuerza * (arr_orig - arr_blur), 0, 255).astype(np.uint8))

    def _optimizar_nitidez_rostros(self, img_pil):
        fino = img_pil.filter(ImageFilter.UnsharpMask(radius=1.2, percent=140, threshold=2))
        pop = ImageEnhance.Contrast(fino).enhance(1.08)
        return ImageEnhance.Brightness(pop).enhance(1.04)

    def _aislar_region_clara_vestido(self, img_original_rgb, alpha_mask):
        arr_alpha = np.array(alpha_mask)
        arr_luma = np.array(img_original_rgb.convert('L'))
        arr_seed = ((arr_alpha > 240) & (arr_luma > 125)).astype(np.uint8) * 255
        return Image.fromarray(arr_seed).filter(ImageFilter.MaxFilter(35))

    def _descontaminar_halos_negros(self, img_rgb, alpha_mask, mascara_vestido_velo):
        arr = np.array(img_rgb).astype(np.float32)
        arr_alpha = np.array(alpha_mask).astype(np.float32) / 255.0
        arr_zona = np.array(mascara_vestido_velo)
        frontera = (arr_alpha > 0.01) & (arr_alpha < 0.96) & (arr_zona > 0)
        factor = (1.0 - arr_alpha) * 0.95
        marfil = [246.0, 244.0, 240.0]
        for i in range(3):
            reconstruido = arr[..., i] + (marfil[i] - arr[..., i]) * factor
            arr[..., i] = np.where(frontera, np.clip(reconstruido, 0, 255), arr[..., i])
        return Image.fromarray(arr.astype(np.uint8))

    def _aplicar_light_wrap(self, composicion_rgb, alpha_mask, fondo_pil, radio=4, intensidad=0.12):
        radio_impar = radio if radio % 2 != 0 else max(3, radio - 1)
        fondo_blur = fondo_pil.filter(ImageFilter.GaussianBlur(radius=radio_impar * 2))
        alpha_encogido = alpha_mask.filter(ImageFilter.MinFilter(radio_impar))
        borde_interno = ImageChops.subtract(alpha_mask, alpha_encogido)
        borde_suave = borde_interno.filter(ImageFilter.GaussianBlur(radius=radio_impar / 2))
        if intensidad < 1.0:
            borde_suave = ImageEnhance.Brightness(borde_suave).enhance(intensidad)
        resultado = composicion_rgb.copy()
        resultado.paste(fondo_blur, (0, 0), borde_suave)
        return resultado

    def revelar_foto_master(self, ruta_entrada, ruta_salida, tema):
        try:
            img_original = Image.open(ruta_entrada).convert('RGB')
            ancho_orig, alto_orig = img_original.size

            # [1] Alpha Matting
            print('   [⏳] [1] Alpha Matting de alta precisión...')
            img_rgba = remove(
                img_original, session=self.session,
                alpha_matting=True,
                alpha_matting_erode_size=3,
                alpha_matting_foreground_threshold=242,
                alpha_matting_background_threshold=12,
            )
            alpha_nativo = img_rgba.split()[3]
            alpha_rasurada = alpha_nativo.filter(ImageFilter.MinFilter(3))
            # [FIX-6] Blur amplio + multiply elimina escaleras en bordes oscuros
            alpha_suavizado = alpha_rasurada.filter(ImageFilter.GaussianBlur(radius=1.2))
            alpha_final = ImageChops.multiply(alpha_suavizado, alpha_suavizado)
            sujetos_rgb = img_rgba.convert('RGB')

            # [6] Fondo FLUX
            ancho_flux, alto_flux = self._calcular_dimensiones_flux(ancho_orig, alto_orig)
            print(f'   [⏳] [6] Generando fondo FLUX ({ancho_flux}x{alto_flux})...')
            fondo_ia = self.client.text_to_image(prompt=self.banco_prompts[tema], width=ancho_flux, height=alto_flux)
            fondo_ia = fondo_ia.resize((ancho_orig, alto_orig), resample=Image.Resampling.LANCZOS)
            fondo_texturizado = self._inyectar_micro_grano_fondo(fondo_ia, intensidad=2.8)

            # [5] Anti-halos vestido/velo
            print('   [⏳] [5] Discriminando áreas textiles para anti-halos...')
            mascara_vestido = self._aislar_region_clara_vestido(img_original, alpha_final)
            sujetos_descontaminados = self._descontaminar_halos_negros(sujetos_rgb, alpha_final, mascara_vestido)

            # [FIX-7] Sharpening + boost de color ANTES de componer
            print('   [⏳] [2] Optimizando nitidez en rostros...')
            sujetos_enfocados = self._optimizar_nitidez_rostros(sujetos_descontaminados)
            print('   [⏳] [3] High Pass micro-detalle estructural...')
            sujetos_high_pass = self._aplicar_high_pass_sharpen(sujetos_enfocados, radio=1.5, fuerza=1.3)
            sujetos_vividos = ImageEnhance.Color(sujetos_high_pass).enhance(1.15)
            sujetos_finales = ImageEnhance.Contrast(sujetos_vividos).enhance(1.06)

            # Composición
            composicion = fondo_texturizado.copy()
            composicion.paste(sujetos_finales, (0, 0), alpha_final)

            # [4] Light Wrap
            print('   [⏳] [4] Light Wrap adaptativo sobre composición...')
            composicion_con_wrap = self._aplicar_light_wrap(composicion, alpha_final, fondo_texturizado, radio=4, intensidad=0.12)

            img_final = ImageEnhance.Contrast(composicion_con_wrap).enhance(1.01)

            # Marco
            grosor = int(alto_orig * 0.035)
            img_entregable = Image.new('RGB', (ancho_orig, alto_orig), (14, 14, 15))
            foto_reducida = img_final.resize((ancho_orig - grosor * 2, alto_orig - grosor * 2), resample=Image.Resampling.LANCZOS)
            img_entregable.paste(foto_reducida, (grosor, grosor))

            # Guardar
            icc = img_original.info.get('icc_profile')
            save_kwargs = {'format': 'JPEG', 'quality': 100, 'subsampling': 0}
            if icc:
                save_kwargs['icc_profile'] = icc
            img_entregable.save(ruta_salida, **save_kwargs)
            return True

        except Exception as e:
            import traceback
            print(f'   ❌ Error: {e}')
            traceback.print_exc()
            return False

    def procesar_lote_master(self, email_cliente, tema):
        carpeta_cliente = os.path.join(self.ruta_base, email_cliente.strip().lower())
        carpeta_upscale = os.path.join(carpeta_cliente, 'pre_upscale')
        carpeta_resultados = os.path.join(carpeta_cliente, 'resultados_finales_x4')

        if not os.path.exists(carpeta_cliente):
            print(f'❌ Carpeta no encontrada: {carpeta_cliente}')
            return

        os.makedirs(carpeta_upscale, exist_ok=True)

        archivos = []
        for ext in ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']:
            archivos.extend(glob.glob(os.path.join(carpeta_cliente, ext)))

        excluir = ['_v', '_WRAP_', '_restored', 'pre_upscale']
        imagenes_validas = [f for f in archivos if not any(v in f for v in excluir)]
        total = len(imagenes_validas)

        if total == 0:
            print('⚠️ No hay archivos originales válidos en la raíz del cliente.')
            return

        print(f'\n⚡ INICIANDO MOTOR V9.8 MASTER — {total} foto(s) ⚡\n')
        exitosos = 0

        for indice, ruta_original in enumerate(imagenes_validas, start=1):
            nombre = os.path.basename(ruta_original)
            nombre_base, _ = os.path.splitext(nombre)
            ruta_salida = os.path.join(carpeta_upscale, f'{nombre_base}_{tema}_v98.jpg')
            print(f'⏳ [{indice}/{total}] Procesando: {nombre}')
            if self.revelar_foto_master(ruta_original, ruta_salida, tema):
                print(f'   ✅ Guardado en: pre_upscale/')
                exitosos += 1
            else:
                print(f'   ⚠️ Omitido por error.')

        print(f'\n📊 Resultado: {exitosos}/{total} fotos procesadas correctamente.')
        print('\n👉 Ejecuta la Celda 4 para Real-ESRGAN x4')
        self._carpeta_upscale = carpeta_upscale
        self._carpeta_resultados = carpeta_resultados


print('✅ Motor V9.8 definido. Continúa con la Celda 3.')

## CELDA 3 — Procesar fotos del cliente
Cambia `EMAIL_CLIENTE` y `TEMA` según el cliente.

Temas: `finca` · `vogue` · `nostalgia` · `magic` · `cinematic`

In [ ]:
# ── CONFIGURA AQUÍ ──────────────────────────────────────────────────
EMAIL_CLIENTE = 'a@a.com'   # <- carpeta del cliente en Drive
TEMA          = 'cinematic' # <- finca | vogue | nostalgia | magic | cinematic
# ────────────────────────────────────────────────────────────────────

motor = ElAlbumB_V9_8_Master()
motor.procesar_lote_master(EMAIL_CLIENTE, TEMA)

## CELDA 4 — Real-ESRGAN x4 (súper-resolución)
Ejecuta **después** de que termine la Celda 3.

In [ ]:
import os, sys

try:
    carpeta_input  = motor._carpeta_upscale
    carpeta_output = motor._carpeta_resultados
except NameError:
    EMAIL_CLIENTE  = 'a@a.com'
    carpeta_input  = f'/content/drive/MyDrive/ElAlbumB_Clientes/{EMAIL_CLIENTE}/pre_upscale'
    carpeta_output = f'/content/drive/MyDrive/ElAlbumB_Clientes/{EMAIL_CLIENTE}/resultados_finales_x4'

print(f'📂 Input:  {carpeta_input}')
print(f'📂 Output: {carpeta_output}')

# Clonar Real-ESRGAN si no existe
if not os.path.exists('/content/Real-ESRGAN'):
    !git clone https://github.com/xinntao/Real-ESRGAN.git /content/Real-ESRGAN -q
    print('✅ Real-ESRGAN clonado.')

%cd /content/Real-ESRGAN

# Instalar basicsr desde fuente (fix para torchvision moderno)
!{sys.executable} -m pip uninstall basicsr -y -q
!{sys.executable} -m pip install "basicsr @ git+https://github.com/XPixelGroup/BasicSR.git" -q
!{sys.executable} -m pip install facexlib gfpgan -q
!{sys.executable} -m pip install -e . -q

# Descargar pesos si no existen
if not os.path.exists('/content/Real-ESRGAN/weights/RealESRGAN_x4plus.pth'):
    !wget https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P weights/ -q
    print('✅ Pesos descargados.')

os.makedirs(carpeta_output, exist_ok=True)

# Copiar a local para evitar problemas de latencia con Drive
import shutil, glob
os.makedirs('/content/input_local', exist_ok=True)
os.makedirs('/content/output_local', exist_ok=True)
for f in glob.glob(os.path.join(carpeta_input, '*.jpg')) + glob.glob(os.path.join(carpeta_input, '*.png')):
    shutil.copy(f, '/content/input_local/')
print(f'✅ {len(os.listdir("/content/input_local"))} foto(s) copiadas a local.')

print('\n🚀 Iniciando súper-resolución x4...')
!python inference_realesrgan.py \
    -n RealESRGAN_x4plus \
    -i "/content/input_local" \
    -o "/content/output_local" \
    --outscale 4 \
    --face_enhance

# Copiar resultados de vuelta a Drive
resultados = glob.glob('/content/output_local/*')
for f in resultados:
    shutil.copy(f, carpeta_output)

print(f'\n✅ {len(resultados)} foto(s) guardadas en Drive: {carpeta_output}')

## CELDA 5 — Vista previa de resultados

In [ ]:
import glob, os
from PIL import Image
import matplotlib.pyplot as plt

try:
    carpeta_ver = motor._carpeta_resultados
except NameError:
    EMAIL_CLIENTE = 'a@a.com'
    carpeta_ver = f'/content/drive/MyDrive/ElAlbumB_Clientes/{EMAIL_CLIENTE}/resultados_finales_x4'

fotos = glob.glob(os.path.join(carpeta_ver, '*.jpg')) + glob.glob(os.path.join(carpeta_ver, '*.png'))

if not fotos:
    print('⚠️ No se encontraron fotos en:', carpeta_ver)
else:
    print(f'📸 {len(fotos)} foto(s) en resultados_finales_x4:')
    for ruta in fotos[:6]:
        img = Image.open(ruta)
        thumb = img.copy()
        thumb.thumbnail((700, 700))
        plt.figure(figsize=(9, 6))
        plt.imshow(thumb)
        plt.title(os.path.basename(ruta), fontsize=10)
        plt.axis('off')
        plt.tight_layout()
        plt.show()
        print(f'  ✅ {os.path.basename(ruta)} — {img.size[0]}x{img.size[1]} px')